# 입낚 물고기 감지 YOLOv8 학습

**순서대로 ▶ 버튼을 눌러서 실행하세요.**

---
**시작 전 준비:**
1. 상단 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU 선택**
2. Roboflow에서 다운받은 zip 파일을 Google Drive에 업로드
   - Google Drive 접속 → 내 드라이브 → `ipnak_dataset.zip` 이름으로 업로드
---

## 1단계: GPU 확인

In [ ]:
!nvidia-smi
print('\n✅ GPU 확인 완료')

## 2단계: YOLOv8 설치

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('\n✅ YOLOv8 설치 완료')

## 3단계: Google Drive 연결 및 데이터 준비

> 실행하면 Google Drive 연결 허용 팝업이 뜹니다. **허용**을 클릭하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive 연결 완료')

In [ ]:
import os, zipfile

# Google Drive에서 zip 파일 경로
ZIP_PATH = '/content/drive/MyDrive/ipnak_dataset.zip'
DATASET_DIR = '/content/dataset'

# zip 파일 존재 확인
if not os.path.exists(ZIP_PATH):
    print('❌ zip 파일을 찾을 수 없습니다.')
    print(f'   경로: {ZIP_PATH}')
    print('   → Google Drive 내 드라이브에 ipnak_dataset.zip 파일을 업로드하세요.')
else:
    print(f'✅ zip 파일 발견: {os.path.getsize(ZIP_PATH) / 1024 / 1024:.1f} MB')
    
    # 압축 해제
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    print(f'✅ 압축 해제 완료: {DATASET_DIR}')

In [ ]:
# 데이터 구조 확인 및 data.yaml 경로 찾기
import glob

# data.yaml 파일 찾기
yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)
print('발견된 yaml 파일:')
for y in yaml_files:
    print(f'  {y}')

# 이미지 수 확인
train_imgs = glob.glob('/content/dataset/**/train/images/*', recursive=True)
valid_imgs = glob.glob('/content/dataset/**/valid/images/*', recursive=True)
test_imgs  = glob.glob('/content/dataset/**/test/images/*', recursive=True)
print(f'\n학습 이미지: {len(train_imgs)}장')
print(f'검증 이미지: {len(valid_imgs)}장')
print(f'테스트 이미지: {len(test_imgs)}장')

In [ ]:
# data.yaml 내용 확인 및 경로 수정
import yaml

YAML_PATH = yaml_files[0] if yaml_files else None

if YAML_PATH:
    with open(YAML_PATH, 'r') as f:
        content = f.read()
    print('현재 data.yaml 내용:')
    print(content)
else:
    print('❌ data.yaml 파일을 찾지 못했습니다.')

In [ ]:
# data.yaml 경로를 절대경로로 수정 (Colab 환경에 맞게)
dataset_root = os.path.dirname(YAML_PATH)

new_yaml = {
    'path': dataset_root,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['fish']
}

FIXED_YAML = '/content/dataset/data.yaml'
with open(FIXED_YAML, 'w') as f:
    yaml.dump(new_yaml, f, default_flow_style=False, allow_unicode=True)

print('✅ data.yaml 수정 완료:')
with open(FIXED_YAML) as f:
    print(f.read())

## 4단계: YOLOv8 학습

> **약 30분~1시간 소요됩니다.** 브라우저 탭을 닫지 마세요.

In [ ]:
from ultralytics import YOLO

# YOLOv8n (가장 작고 빠른 모델) - 모바일 웹앱에 최적화
model = YOLO('yolov8n.pt')

results = model.train(
    data=FIXED_YAML,
    epochs=100,          # 학습 반복 횟수
    imgsz=640,           # 입력 이미지 크기 (입낚 서버 기준)
    batch=16,            # 배치 크기 (GPU 메모리에 맞게 자동 조정)
    name='ipnak_fish_v1',
    project='/content/runs',
    patience=20,         # 20 epoch 동안 개선 없으면 조기 종료
    save=True,
    device=0,            # GPU 사용
    exist_ok=True,
)

print('\n✅ 학습 완료!')
print(f'최고 모델: {results.save_dir}/weights/best.pt')

## 5단계: ONNX 변환

> 입낚 서버에서 사용할 수 있는 형식으로 변환합니다.

In [ ]:
import os

# 학습된 최고 모델 불러오기
best_pt = '/content/runs/ipnak_fish_v1/weights/best.pt'
model = YOLO(best_pt)

# ONNX 변환
model.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
)

onnx_path = best_pt.replace('.pt', '.onnx')
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'\n✅ ONNX 변환 완료!')
print(f'파일: {onnx_path}')
print(f'크기: {size_mb:.1f} MB')

## 6단계: best.onnx 다운로드

> 이 셀을 실행하면 **best.onnx 파일이 자동으로 다운로드**됩니다.

In [ ]:
from google.colab import files

onnx_path = '/content/runs/ipnak_fish_v1/weights/best.onnx'
files.download(onnx_path)
print('✅ 다운로드 시작됨!')
print('\n다음 단계: 입낚 관리자 → AI 학습관리 → 모델 관리 탭에 업로드하세요.')

---
## 완료!

다운받은 `best.onnx` 파일을:
1. 입낚 관리자 페이지 접속
2. AI 학습관리 → **모델 관리** 탭
3. **파일 선택** → best.onnx 선택
4. **업로드 후 즉시 적용** 클릭

완료되면 입낚 AI 카메라가 YOLO 모델로 물고기를 감지합니다.